In [1]:
# Imports and base setup
import os
import re
import json
import hashlib
from typing import Dict, List, Tuple

# Must be set BEFORE importing huggingface_hub/transformers
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Patch SSL verification BEFORE importing transformers/huggingface
import urllib3
urllib3.disable_warnings()

# Patch HTTPX to disable SSL verification
try:
    import httpx
    httpx._verify_disabled = True
except ImportError:
    pass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from huggingface_hub import login

c:\projects\learn-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [3]:
# Load text from data.txt
from pathlib import Path

candidates = [
    Path.cwd() / "data.txt",
    Path.cwd().parent / "data.txt",
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("data.txt not found in current directory or parent directory.")

text_data = data_path.read_text(encoding="utf-8")
print(f"Loaded {len(text_data)} characters from {data_path}")
print(text_data[:500])

Loaded 50293 characters from c:\projects\learn-rag\Retrieval_and_reranking\data.txt
The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in


In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Ensure Xet Storage is disabled (must also be set before huggingface_hub import in cell 3)
os.environ["HF_HUB_DISABLE_XET"] = "1"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
token_arg = hf_token if hf_token else None

model = AutoModelForSequenceClassification.from_pretrained(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    # use_auth_token=token_arg
)
tokenizer = AutoTokenizer.from_pretrained(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    # use_auth_token=token_arg
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3499.50it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# Authenticate with Hugging Face - Configure SSL & disable problematic features
print("=" * 60)
print("Setting up Hugging Face Authentication...")
print("=" * 60)

# Disable Xet Storage (causes download issues)
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Patch HTTPX to disable SSL verification
try:
    import ssl
    import httpx
    
    original_client_init = httpx.Client.__init__
    
    def patched_init(self, *args, **kwargs):
        kwargs['verify'] = False
        return original_client_init(self, *args, **kwargs)
    
    httpx.Client.__init__ = patched_init
    print("✓ Network configuration set")
except Exception as e:
    print(f"Note: {type(e).__name__}")

# Suppress warnings  
import urllib3
urllib3.disable_warnings()

# Clear HF cache if it's causing issues
import shutil
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(hf_cache):
    try:
        print("Cleaning HF cache...")
        shutil.rmtree(hf_cache)
        print("✓ HF cache cleared")
    except Exception as e:
        print(f"Note: Could not clear cache - {e}")

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")

print(f"\n1. HF_TOKEN Status:")
if not hf_token:
    print("   WARNING: HF_TOKEN not found in .env")
else:
    print(f"   ✓ Found (length: {len(hf_token)} chars)")

print(f"\n2. Configuration:")
print("   - Xet Storage disabled")
print("   - SSL verification disabled")
print("   - Cache cleared")
print("=" * 60)

Setting up Hugging Face Authentication...
✓ Network configuration set
Cleaning HF cache...
✓ HF cache cleared

1. HF_TOKEN Status:
   ✓ Found (length: 37 chars)

2. Configuration:
   - Xet Storage disabled
   - SSL verification disabled
   - Cache cleared


<h2>Sample ReRanking checking Scores between 2 Sentences</h2>

<h4>Later On will Test this Question & Answer<h4>

In [5]:
features = tokenizer(
    ['How many people live in Berlin?', 'How many people live in Berlin?'],
    ['Berlin has a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.',
     'New York City is famous for the Metropolitan Museum of Art.'],
    padding=True, truncation=True, return_tensors="pt"
)

model.eval()
with torch.no_grad():
    scores = model(**features).logits
    print(scores)

tensor([[  8.8459],
        [-11.2456]])


<h4>Now I will Normalize This scores and make it into probabilities</h4>

In [6]:
import torch

def normalize_scores(scores):
    return torch.sigmoid(torch.tensor(scores)).tolist()

# Example
raw_scores = scores
probs = normalize_scores(raw_scores)
probs

C:\Users\veguntur\AppData\Local\Temp\ipykernel_44880\674195476.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.sigmoid(torch.tensor(scores)).tolist()


[[0.9998559951782227], [1.30649868879118e-05]]

RecursiveCharacterTextSplitter/Fixed-length chunking 

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
texts = text_splitter.split_text(text_data)

In [8]:
texts

['The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th',
 'teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer',
 'window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]',
 'In 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports l

SpacyTextSplitter - sentence splitting

In [16]:
from langchain_text_splitters import SpacyTextSplitter

text_splitter = SpacyTextSplitter(
    pipeline="sentencizer",
    chunk_size=500,
    chunk_overlap=75,
)

texts = text_splitter.split_text(text_data)

Created a chunk of size 877, which is longer than the specified 500
Created a chunk of size 917, which is longer than the specified 500
Created a chunk of size 569, which is longer than the specified 500
Created a chunk of size 535, which is longer than the specified 500
Created a chunk of size 2017, which is longer than the specified 500
Created a chunk of size 1630, which is longer than the specified 500
Created a chunk of size 1068, which is longer than the specified 500
Created a chunk of size 794, which is longer than the specified 500
Created a chunk of size 2105, which is longer than the specified 500
Created a chunk of size 4679, which is longer than the specified 500
Created a chunk of size 1771, which is longer than the specified 500
Created a chunk of size 2464, which is longer than the specified 500
Created a chunk of size 1725, which is longer than the specified 500
Created a chunk of size 1685, which is longer than the specified 500
Created a chunk of size 1127, which is 

In [17]:
texts

['The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue.\n\nIt is held annually between March and May.',
 'It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]\n\nIn 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of the IPL, other Indian sports leagues have been established.[a][11][12] The IPL is the second-richest sports league in the world by per-match value, a

In [24]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', token=token_arg)

c:\projects\learn-rag\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\veguntur\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2656.05it/s]
BertForSequenceClassification LOA

Since the Texts are small We have directly sent the whole data else we will use a Vector DB

That function takes:

One question
A list of candidate answers (or chunks)
Then it:

Scores each question-answer pair with the cross-encoder
Converts scores to comparable relevance values
Sorts them highest to lowest
Returns top K


In [25]:
import torch

def top_k_rerank(question, answers, top_k=5):
    """Return top-k candidate answers ranked by cross-encoder relevance score."""
    if not answers:
        return []

    top_k = max(1, min(top_k, len(answers)))

    features = tokenizer(
        [question] * len(answers),
        answers,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    model.eval()
    with torch.no_grad():
        logits = model(**features).logits.squeeze(-1)

    scores = torch.sigmoid(logits).tolist()

    ranked = sorted(
        [{"answer": ans, "score": float(score)} for ans, score in zip(answers, scores)],
        key=lambda x: x["score"],
        reverse=True,
    )
    return ranked[:top_k]

# Example usage
question = "What is IPL and who organizes it?"
top_results = top_k_rerank(question, texts, top_k=3)
top_results

[{'answer': 'The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue.\n\nIt is held annually between March and May.',
  'score': 0.9993213415145874},
 {'answer': "BCCI vice-president Lalit Modi, who led the IPL initiative, provided details of the tournament, including its format, prize money, franchise revenue system, and squad composition rules.\n\nThe league, to be managed by a seven-person governing council, would also serve as the qualifying mechanism for that year's Champions League Twenty20.[24][25]\n\nTo determine team ownership, an auction for the franchises was held on 24 January 2008.",
  'score': 0.8757461905479431},
 {'answer': 'It has an exclusive window in the Future Tours Programme o